# Unsloth QLoRA Fine-Tuning + Evaluation

## Environment Setup

In [ ]:
# ! pip install python-dotenv

from dotenv import load_dotenv
load_dotenv()

import os

# Set env vars FIRST
os.environ["HF_HOME"] = "/tmp/mawojide/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/mawojide/hf_cache/transformers"
os.environ["HF_DATASETS_CACHE"] = "/tmp/mawojide/hf_cache/datasets"
os.environ["HF_TEMP_DIR"] = "/tmp/huggingface_temp"

os.makedirs("/tmp/mawojide/hf_cache/transformers", exist_ok=True)
os.makedirs("/tmp/mawojide/hf_cache/datasets", exist_ok=True)

print("HF_HOME:", os.environ["HF_HOME"])  # confirm it's set

from huggingface_hub import constants
print(constants.HF_HUB_CACHE)

from huggingface_hub import login
from huggingface_hub import HfApi

# 3. Explicitly log in for this session
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=True)
print("Login attempt finished.")


api = HfApi()
try:
    user = api.whoami()
    print(f"Logged in as: {user['name']}")
    print(f"Token Role: {user['auth']['accessToken']['role']}") 
    # Must say 'write'
except Exception as e:
    print(f"Login failed: {e}")

In [ ]:
# Install Unsloth + dependencies
# Unsloth auto-detects CUDA version and installs matching wheels
# ! pip install "unsloth[cu121-ampere-torch230]" -q   # CUDA 12.1, Ampere/H100
# ! pip install unsloth -q                             # generic fallback
# ! pip install scikit-learn plotly -q                 # evaluation deps
# ! python -m pip install wandb weave -q         # logging deps


In [ ]:
import os, re, math, statistics
from datetime import datetime
from itertools import accumulate
from IPython.display import clear_output

import torch
from unsloth import FastLanguageModel           # replaces AutoModelForCausalLM
from trl import SFTTrainer
from transformers import TrainingArguments, set_seed
from datasets import load_dataset
from huggingface_hub import login

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

print("torch         :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU           :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
capability = torch.cuda.get_device_capability()
USE_BF16   = capability[0] >= 8   # True on H100 / A100


## Constants & Hyperparameters

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL   = "unsloth/Qwen2.5-Math-1.5B-bnb-4bit"
PROJECT_NAME = "price"
HF_USER      = "martinsawojide"

# ── Dataset ───────────────────────────────────────────────────────────────────
LITE_MODE    = False
DATA_USER    = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# ── Run naming ────────────────────────────────────────────────────────────────
RUN_NAME         = f"{datetime.now():%Y-%m-%d_%H.%M.%S}" + ("-lite" if LITE_MODE else "")
PROJECT_RUN_NAME = f"/tmp/checkpoint_dir/{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME   = f"{HF_USER}/{PROJECT_NAME}-{RUN_NAME}"

# ── Sequence length ───────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 256

# ── QLoRA ----------------------------─────────────────────────────────────────
LORA_R       = 16 if LITE_MODE else 64
LORA_ALPHA   = LORA_R * 2     
LORA_DROPOUT = 0              
ATTENTION_LAYERS = ["q_proj", "k_proj", "v_proj", "o_proj"]
MLP_LAYERS       = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES   = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS          = 1   if LITE_MODE else 3
BATCH_SIZE      = 8  if LITE_MODE else 128
GRAD_ACCUM      = 2                    
LEARNING_RATE   = 1e-4                 
WARMUP_STEPS    = 50  if LITE_MODE else 100
LR_SCHEDULER    = "cosine"
WEIGHT_DECAY    = 0.05                
OPTIMIZER       = "adamw_8bit"
MAX_GRAD_NORM   = 0.3

# ── Dataset ──────────────────────────────────────────────────────────
dataset   = load_dataset(DATASET_NAME)
train     = dataset["train"]
val       = dataset["val"]  
test      = dataset["test"]

print(f"Train : {len(train):,}")
print(f"Val   : {len(val):,}")
print(f"Test  : {len(test):,}")

# ── Logging / saving ──────────────────────────────────────────────────────────
LOG_STEPS    = 5    if LITE_MODE else 25
SAVE_STEPS   = 100  if LITE_MODE else 250
VAL_SIZE     = 500  if LITE_MODE else len(val)
LOG_TO_WANDB = True 

# ── Inference Optimization (best config validated through experiments) ───────
BEAM_NUM_BEAMS   = 10
BEAM_NUM_GROUPS  = 10
BEAM_DIV_PENALTY = 1.5
EVAL_SIZE        = len(test)

print(f"Run name       : {RUN_NAME}")
print(f"Hub model      : {HUB_MODEL_NAME}")
print(f"Dataset        : {DATASET_NAME}")
print(f"Max seq length : {MAX_SEQ_LENGTH}")
print(f"LoRA r / alpha : {LORA_R} / {LORA_ALPHA}")
print(f"Batch / epochs : {BATCH_SIZE} / {EPOCHS}")
print(f"Optimizer      : {OPTIMIZER}")


In [ ]:
if LOG_TO_WANDB:
    import wandb
    os.environ["WANDB_API_KEY"]   = os.getenv("WANDB_API_KEY", "")
    os.environ["WANDB_PROJECT"]   = PROJECT_NAME
    os.environ["WANDB_LOG_MODEL"] = "false"
    os.environ["WANDB_WATCH"]     = "false"
    wandb.login()

## Load Base Model with Unsloth


In [ ]:
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,   
    load_in_4bit   = True,             
)

# Unsloth requires pad_token to be set for generation
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
base_model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded  : {BASE_MODEL}")
print(f"Memory        : {base_model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Pad token id  : {tokenizer.pad_token_id}")


## Inference Helpers


In [ ]:
def extract_price(text: str):
    """Extract the first plausible USD price from generated text."""
    patterns = [
        r"\$[\d,]+\.?\d*",   # $12.99  $1,299
        r"[\d,]+\.\d{2}",    # 12.99
        r"[\d,]+",             # 12
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            price_str = match.group().replace("$", "").replace(",", "")
            try:
                price = float(price_str)
                if 0.01 <= price <= 100_000:
                    return price
            except ValueError:
                continue
    return None


def predict_with_model(model, item,
                       num_beams=BEAM_NUM_BEAMS,
                       num_groups=BEAM_NUM_GROUPS,
                       diversity_penalty=BEAM_DIV_PENALTY):
    """
    Diverse beam search → geometric mean of all beam prices.
    Best config found: 10 beams / 5 groups / penalty=1.0 → 86.7 MAE on base model.
    Works with both base model and fine-tuned Unsloth model.
    """
    # Switch to Unsloth's fast inference kernels
    FastLanguageModel.for_inference(model)

    prompt       = str(item["prompt"])
    inputs       = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens      = 8,
            num_beams           = num_beams,
            num_beam_groups     = num_groups,
            diversity_penalty   = diversity_penalty,
            num_return_sequences= num_beams,
            pad_token_id        = tokenizer.eos_token_id,
            use_cache           = False,    # Unsloth requires this for KV cache speedup but it is buggy
            trust_remote_code   = True,    # required for Unsloth models from HF Hub
            custom_generate='transformers-community/group-beam-search', # handles error messages for beam search 
        )

    prices = []
    for beam_output in output_ids:
        text  = tokenizer.decode(beam_output[input_length:], skip_special_tokens=True)
        price = extract_price(text)
        if price:
            prices.append(price)

    if not prices:
        return None
    return statistics.geometric_mean(prices)


# Named wrapper so Tester.make_title() generates a clean label
def base_model_predict(item):
    return predict_with_model(base_model, item)


## Evaluation Framework (Tester + Plotly Charts)

In [ ]:
GREEN     = "\033[92m"
YELLOW    = "\033[93m"
RED       = "\033[91m"
RESET     = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}


class Tester:
    def __init__(self, predictor, data, title=None, size=EVAL_SIZE):
        self.predictor = predictor
        self.data      = data
        self.title     = title or self.make_title(predictor)
        self.size      = size
        self.titles    = []
        self.guesses   = []
        self.truths    = []
        self.errors    = []
        self.colors    = []

    @staticmethod
    def make_title(predictor) -> str:
        return predictor.__name__.replace("__", ".").replace("_", " ").title().replace("Gpt", "GPT")

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", value)
            return float(match.group()) if match else 0
        return value if value is not None else 0

    def color_for(self, error, truth):
        if error < 40 or error / truth < 0.2:    return "green"
        elif error < 80 or error / truth < 0.4:  return "orange"
        else:                                     return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        value     = self.predictor(datapoint)
        guess     = self.post_process(value)
        truth     = float(datapoint["completion"])
        error     = abs(guess - truth)
        color     = self.color_for(error, truth)
        pieces    = datapoint["prompt"].split("Title: ")
        title     = pieces[1].split("\n")[0] if len(pieces) > 1 else pieces[0]
        title     = title if len(title) <= 40 else title[:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame({
            "truth": self.truths, "guess": self.guesses,
            "title": self.titles, "error": self.errors, "color": self.colors,
        })
        df["hover"] = [
            f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]
        max_val = float(max(df["truth"].max(), df["guess"].max()))
        fig = px.scatter(
            df, x="truth", y="guess", color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title, labels={"truth": "Actual Price", "guess": "Predicted Price"},
            width=800, height=600,
        )
        for tr in fig.data:
            mask         = df["color"] == tr.name
            tr.customdata   = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)
        fig.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val], mode="lines",
            line=dict(width=2, dash="dash", color="deepskyblue"),
            hoverinfo="skip", showlegend=False,
        ))
        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n               = len(self.errors)
        running_sums    = list(accumulate(self.errors))
        x               = list(range(1, n + 1))
        running_means   = [s / i for s, i in zip(running_sums, x)]
        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds    = [
            math.sqrt((sq / i) - (m ** 2)) if i > 1 else 0
            for i, sq, m in zip(x, running_squares, running_means)
        ]
        ci    = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]
        title = f"{self.title}  |  Final MAE: ${running_means[-1]:,.2f} ± ${ci[-1]:,.2f}"
        fig   = go.Figure()
        fig.add_trace(go.Scatter(
            x=x + x[::-1], y=upper + lower[::-1], fill="toself",
            fillcolor="rgba(128,128,128,0.2)",
            line=dict(color="rgba(255,255,255,0)"), hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=x, y=running_means, mode="lines",
            line=dict(width=3, color="firebrick"),
            customdata=list(zip(ci)),
            hovertemplate="n=%{x}<br>Avg Error=$%{y:,.2f}<br>±95% CI=$%{customdata[0]:,.2f}<extra></extra>",
        ))
        fig.update_layout(
            title=title, xaxis_title="Number of Datapoints",
            yaxis_title="MAE ($)", width=800, height=300,
            template="plotly_white", showlegend=False,
        )
        fig.show()

    def report(self):
        avg_error = sum(self.errors) / self.size
        mse       = mean_squared_error(self.truths, self.guesses)
        r2        = r2_score(self.truths, self.guesses) * 100
        title     = (f"{self.title}<br>"
                     f"<b>MAE:</b> ${avg_error:,.2f}  "
                     f"<b>MSE:</b> {mse:,.0f}  "
                     f"<b>r²:</b> {r2:.1f}%")
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        for i in tqdm(range(self.size), desc=self.title):
            title, guess, truth, error, color = self.run_datapoint(i)
            self.titles.append(title)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)
            print(f"{COLOR_MAP[color]}${error:.0f} ", end="")
        clear_output(wait=True)
        self.report()
        return self  # allow chaining


def evaluate(function, data, size=EVAL_SIZE):
    return Tester(function, data, size=size).run()


## Prepare Dataset for SFT

In [ ]:
def format_for_sft(examples):
    """
    Combine prompt + completion into a single text field.
    completion is the price string e.g. '12.99' — append EOS so model learns to stop.
    """
    texts = []
    for prompt, completion in zip(examples["prompt"], examples["completion"]):
        # Full sequence: prompt already ends with 'Price is $', completion is the number
        full_text = str(prompt) + str(completion) + tokenizer.eos_token
        texts.append(full_text)
    return {"text": texts}


# Map in batches — fast, avoids Python loop overhead
train_sft = train.map(format_for_sft, batched=True, remove_columns=train.column_names)
val_sft   = val.map(format_for_sft,   batched=True, remove_columns=val.column_names)

# Verify
print("Sample SFT text (first 200 chars):")
print(train_sft[0]["text"][:200])
print(f"\nTrain SFT size : {len(train_sft):,}")
print(f"Val SFT size   : {len(val_sft):,}")


## Add LoRA Adapters with Unsloth


In [ ]:
base_model.train() # confirm we are in training mode

base_model = FastLanguageModel.get_peft_model(
    base_model,
    r                   = LORA_R,
    target_modules      = TARGET_MODULES,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,   # 0 — Unsloth recommendation
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # 30% more context, same VRAM
    random_state        = 3407,
    use_rslora          = True,          # rank-stabilised LoRA — set True for r >= 128
    loftq_config        = None,
)

# Count trainable parameters
trainable   = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total       = sum(p.numel() for p in base_model.parameters())
print(f"Trainable params : {trainable:,}  ({100 * trainable / total:.2f}% of total)")
print(f"Total params     : {total:,}")
print(f"LoRA rank        : {LORA_R}  |  alpha: {LORA_ALPHA}  |  dropout: {LORA_DROPOUT}")
print(f"Target modules   : {TARGET_MODULES}")


## Fine-Tune with Unsloth SFTTrainer


In [ ]:
train_args = TrainingArguments(
    # ── Output ───────────────────────────────────────────────────────────────
    output_dir          = PROJECT_RUN_NAME,

    # ── Training duration ────────────────────────────────────────────────────
    num_train_epochs    = EPOCHS,
    max_steps           = -1,           # -1 = run full epochs

    # ── Batch & gradient ─────────────────────────────────────────────────────
    per_device_train_batch_size  = BATCH_SIZE,
    per_device_eval_batch_size   = 1,
    gradient_accumulation_steps  = GRAD_ACCUM,

    # ── Optimiser & LR ───────────────────────────────────────────────────────
    optim               = OPTIMIZER,    
    learning_rate       = LEARNING_RATE,
    weight_decay        = WEIGHT_DECAY,
    warmup_steps        = WARMUP_STEPS,
    lr_scheduler_type   = LR_SCHEDULER,
    max_grad_norm       = MAX_GRAD_NORM,

    # ── Precision --------─────────────────────────────────────────────────────
    fp16                = False,
    bf16                = USE_BF16,     

    # ── Logging ──────────────────────────────────────────────────────────────
    logging_steps       = LOG_STEPS,
    report_to           = "wandb" if LOG_TO_WANDB else "none",
    run_name            = RUN_NAME,

    # ── Checkpointing ────────────────────────────────────────────────────────
    save_strategy       = "steps",
    save_steps          = SAVE_STEPS,
    save_total_limit    = 2,            # keep last 2 checkpoints to save disk

    # ── Evaluation ───────────────────────────────────────────────────────────
    eval_strategy       = "steps",
    eval_steps          = SAVE_STEPS,

    # ── Misc ─────────────────────────────────────────────────────────────────
    seed                    = 3407,
    dataloader_num_workers  = 4,              # H100 has CPU headroom to parallelise data loading
    save_only_model         = True,           # save only LoRA weights to save disk and speed up hub push   
    load_best_model_at_end  = False,
    push_to_hub             = False,
)

print(f"Output dir     : {PROJECT_RUN_NAME}")
print(f"Hub target     : {HUB_MODEL_NAME}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")


In [ ]:
trainer = SFTTrainer(
    model               = base_model,
    tokenizer           = tokenizer,
    train_dataset       = train_sft,
    eval_dataset        = val_sft,
    dataset_text_field  = "text",           # the column we created in format_for_sft
    max_seq_length      = MAX_SEQ_LENGTH,
    packing             = True,             # critical for throughput win
    args                = train_args,
)

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
print(f"Trainer ready — {trainable:,} trainable parameters")


## Start training

In [ ]:
if LOG_TO_WANDB:
    import wandb
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Total steps    : {trainer_stats.global_step}")
print(f"Training loss  : {trainer_stats.training_loss:.4f}")
print(f"Runtime        : {trainer_stats.metrics['train_runtime'] / 60:.1f} min")

if LOG_TO_WANDB:
    wandb.finish()


## Save & Push Final Model

In [ ]:
# Save LoRA adapter weights only
trainer.model.save_pretrained(PROJECT_RUN_NAME)
tokenizer.save_pretrained(PROJECT_RUN_NAME)
print(f"Saved locally  → {PROJECT_RUN_NAME}/")

# Push final adapter to Hub
trainer.model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Pushed to Hub  → {HUB_MODEL_NAME}")


## Load Fine-Tuned Model for Inference


In [ ]:
# ── If loading a previously saved run instead of the one just trained: ─────────
# HUB_MODEL_NAME = "martinsawojide/price-YYYY-MM-DD_HH.MM.SS"
HUB_MODEL_NAME = "martinsawojide/price-2026-03-06_17.59.28"

fine_tuned_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = HUB_MODEL_NAME,   # adapter + base weights loaded together
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,
    load_in_4bit   = True,
)

# Activate Unsloth's fast inference kernels — required before generation
FastLanguageModel.for_inference(fine_tuned_model)

# Ensure pad token matches training setup
ft_tokenizer.pad_token    = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "right"
fine_tuned_model.config.pad_token_id = ft_tokenizer.eos_token_id

print(f"Fine-tuned model loaded : {HUB_MODEL_NAME}")
print(f"Memory footprint        : {fine_tuned_model.get_memory_footprint() / 1e9:.2f} GB")


## Fine-Tuned Model Evaluation

In [ ]:
def fine_tuned_predict(item):
    """Wrapper using fine-tuned model — for_inference already called above."""
    prompt       = str(item["prompt"])
    inputs       = ft_tokenizer(prompt, return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = fine_tuned_model.generate(
            **inputs,
            max_new_tokens       = 8,
            num_beams            = BEAM_NUM_BEAMS,
            num_beam_groups      = BEAM_NUM_GROUPS,
            diversity_penalty    = BEAM_DIV_PENALTY,
            num_return_sequences = BEAM_NUM_BEAMS,
            pad_token_id         = ft_tokenizer.eos_token_id,
            use_cache            = False,    
            trust_remote_code    = True,     
            custom_generate='transformers-community/group-beam-search',
        )

    prices = []
    for beam_output in output_ids:
        text  = ft_tokenizer.decode(beam_output[input_length:], skip_special_tokens=True)
        price = extract_price(text)
        if price:
            prices.append(price)

    return statistics.geometric_mean(prices) if prices else None


In [ ]:
set_seed(42)
ft_tester = evaluate(fine_tuned_predict, test, size=EVAL_SIZE)
